# LifeMate — finetune QLoRA trên Kaggle (2× T4)Notebook này **chỉ** huấn luyện adapter. Suy luận và chấm điểm nằm ở`lifemate_kaggle_v2.ipynb`, chạy sau khi có adapter.Trước đây nó chỉ tồn tại trên Kaggle, không có trong repo — nên hai lượt chết(4 giờ 12 phút và 5 phút) không để lại vết nào để xem lại. Bản này dựng lại từlog của lượt chạy, **không phải bản sao** notebook cũ, nên hãy đối chiếu phần`ARGS` với thứ bạn thực sự muốn chạy.## Cần gì trong Input- Dataset `lifemate-sft` chứa: `train_sft.py`, `sft.train.jsonl`, `sft.eval.jsonl`- Accelerator: **GPU T4 x2**, Internet **On** (để tải trọng số lần đầu)## Lượt chạy320 mẫu, `grad-accum 4` → 80 bước. Đo được 298 giây/bước → **~6.6 giờ**.`--save-steps 10` nên nếu đứt thì mất tối đa ~50 phút; chạy lại y nguyên lệnhnày, `--resume` tự tiếp từ checkpoint mới nhất.

In [ ]:
# 1. Phien ban, duong dan, va kiem tra Input truoc khi lam gi ton thoi gianimport sys, subprocess, importlib.metadata as mdfrom pathlib import Pathfor pkg in ("transformers", "accelerate", "peft", "bitsandbytes", "torch"):    try:        print(f"{pkg:14} {md.version(pkg)}")    except md.PackageNotFoundError:        print(f"{pkg:14} THIEU")# Do bang glob, KHONG duong dan cung: dataset mount o /kaggle/input/datasets/<user>/<slug>/# chu khong phai /kaggle/input/<slug>/. Duong cung sai tung lam preflight lay mau tu ca split# va lam nhiem diem -- xem EXPERIMENTS.md.def find_one(pattern: str) -> Path:    hits = sorted(Path("/kaggle/input").glob(pattern))    if not hits:        raise FileNotFoundError(f"khong thay {pattern} trong /kaggle/input -- da Add Data chua?")    return hits[0]SCRIPT = find_one("**/train_sft.py")TRAIN  = find_one("**/sft.train.jsonl")EVAL   = find_one("**/sft.eval.jsonl")OUT    = Path("/kaggle/working/lora_V3")print()for name, p in (("script", SCRIPT), ("train", TRAIN), ("eval", EVAL)):    print(f"{name:7} {p}")# In ngay ban train_sft.py dang dung. Hai luot truoc chet vi script tren dataset KHAC voi# script trong repo; mot dong hash o day la cach duy nhat doc log ma biet minh chay ban nao.import hashlibprint()print(f"sha256(train_sft.py)[:16] = {hashlib.sha256(SCRIPT.read_bytes()).hexdigest()[:16]}")print(f"so dong = {len(SCRIPT.read_text(encoding='utf-8').splitlines())}")

In [ ]:
# 2. Tai trong so (chi lan dau; sau do nam trong /root/.cache nen chay lai la mien phi)from huggingface_hub import snapshot_downloadLLM_ID  = "Qwen/Qwen3-4B-Instruct-2507"LLM_REV = "cdbee75f17c01a7cc42f958dc650907174af0554"# Ban INSTRUCT, khong phai Thinking: quy dao gold khong co khoi <think> nao, va adapter hoc tren# trong so Instruct khong nap duoc lenlen trong so Thinking. Doi cho nay la doi ca huong du an.MODEL = snapshot_download(LLM_ID, revision=LLM_REV,                          allow_patterns=["*.json", "*.safetensors", "*.jinja", "*.txt"])print("model :", MODEL)

In [ ]:
# 3. Huan luyen. Moi co o day deu co mot luot chay chet dang sau no.ARGS = [    "--train", str(TRAIN), "--eval", str(EVAL),    "--model", MODEL,    "--out", str(OUT),    "--attn", "block",        # khong dung ma tran (1,32,L,L) tren sm_75    "--quant", "fp16",        # 7.49 GiB, vua MOT card T4 (14.56 GiB)    "--device", "single",     # xem ghi chu ben duoi    "--limit", "320", "--eval-limit", "40",    "--grad-accum", "4",      # 320/4 = 80 buoc    "--save-steps", "10",     # luu theo BUOC: mot luot chet o buoc 46/80 sau 4h12 da mat trang    "--resume",               # chay lai y nguyen lenh nay thi tu tiep tu checkpoint moi nhat    "--epochs", "1",    "--max-seq-len", "8192",  # 8192 giu 100% hoi thoai (dai nhat 8181 token); 4096 chi giu 18.8%    "--cache", "/tmp/sft.tok",]# --device single: dat trong so len mot card de bo han hook chuyen tensor cua accelerate, cho no# illegal memory access o buoc 46/80. Nhung Trainer KHONG doc device_map -- no doc# torch.cuda.device_count(), thay 2 card, dat n_gpu=2 roi boc nn.DataParallel, lam luot sau chet# ngay buoc 0 voi "khong tim thay lm_head tren DataParallel". train_sft.py gio ghim n_gpu=1;# log phai in dong "ghim n_gpu=1: khong boc DataParallel", neu khong thay thi script tren dataset# la ban CU -- dung de no chay tiep, cap nhat dataset truoc.# Khong can --vram-fraction: chot chan VRAM tu bo qua tren card >= 8 GiB.print(" ".join(["python", SCRIPT.name] + ARGS), flush=True)proc = subprocess.run([sys.executable, str(SCRIPT)] + ARGS)print("returncode =", proc.returncode)if proc.returncode != 0:    raise SystemExit("train_sft.py that bai -- doc traceback o tren truoc khi chay lai")

In [ ]:
# 4. Adapter da ra chua, va nang bao nhieufor p in sorted(OUT.rglob("*")):    if p.is_file():        print(f"{p.stat().st_size / 1024**2:9.2f} MB  {p.relative_to(OUT)}")print("Tai ca thu muc lora_V3 ve, roi dat vao mot Kaggle Dataset moi de"      " lifemate_kaggle_v2.ipynb tro ADAPTER_DIR vao do.")